# Day 5: พยากรณ์ค่ามลพิษล่วงหน้า 24 ชั่วโมง

ต่อจาก `04_attribution.ipynb` โดยใช้ `data/processed/hourly_flagged.parquet` และข้อมูลลม `data/raw/wind_hourly_raw.csv`

**เป้าหมาย:** ณ เวลา t ทายค่า PM2.5 และ SO2 ที่เวลา t+24 ของสถานี 225611 (ใกล้นิคมที่สุด)

**สิ่งที่ทำ**
1. เตรียมตารางรายชั่วโมงให้ครบทุกชั่วโมง
2. สร้างฟีเจอร์ที่ใช้เฉพาะข้อมูล ณ เวลา t หรือก่อนหน้า (กันข้อมูลรั่ว)
3. แบ่งข้อมูลตามเวลา ไม่สุ่ม
4. baseline: persistence และ seasonal naive
5. LightGBM เทียบกับ baseline ด้วย MAE / RMSE

## 5.1 เตรียมตารางรายชั่วโมง

**ทำไมต้องเติมแถวให้ครบทุกชั่วโมง:** ข้อมูลมีช่องว่างเยอะ ถ้าใช้ `shift(24)` กับตารางที่ขาดแถว จะได้ค่าของ "24 แถวก่อน" ซึ่งอาจเป็น 3 วันก่อน ไม่ใช่ 24 ชั่วโมงก่อน
เติมแถวที่ขาดเป็นค่าว่าง (NaN) ก่อน แล้ว `shift` จะนับเป็นชั่วโมงจริง

**ลมดึงจากไฟล์ดิบ** ไม่ใช้ลมใน `hourly_flagged.parquet` เพราะไฟล์นั้นมีลมเฉพาะชั่วโมงที่เครื่องวัดอากาศมีข้อมูล (Day 2 merge แบบยึดตารางอากาศ)
แต่ลมจาก Open-Meteo มีครบทุกชั่วโมง ใช้ IFS เป็นหลักเหมือน Day 4 (ช่วงข้อมูลของ 225611 ได้ลม IFS ครบ 100% ตามที่ตรวจใน Day 2)

**สถานีพื้นหลัง** ใช้ PM2.5 ของ 717 อย่างเดียว เพราะ SO2 ของ 717 หยุดตั้งแต่ปลายปี 2024 ซึ่งเป็นช่วงก่อนข้อมูลทดสอบ

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

STATION, BACKGROUND = 225611, 717
WIND_VARS = ['wind_speed_10m', 'wind_direction_10m', 'precipitation', 'temperature_2m', 'relative_humidity_2m']

flagged = pd.read_parquet('../data/processed/hourly_flagged.parquet')

# ค่ามลพิษของสถานีหลัก
main = flagged[flagged.station_id == STATION]
main = main.set_index('datetime_utc').sort_index()
main = main[['pm25', 'so2']]

# เติมแถวให้ครบทุกชั่วโมง ชั่วโมงที่ไม่มีข้อมูลจะเป็น NaN
data = main.asfreq('h')

# PM2.5 ของสถานีพื้นหลัง 717 ที่ชั่วโมงเดียวกัน
bg = flagged[flagged.station_id == BACKGROUND].set_index('datetime_utc')
data['pm25_bg'] = bg['pm25']

# ลม IFS ที่ตำแหน่งสถานีหลัก จากไฟล์ดิบ (Open-Meteo ให้มาเป็น UTC แต่ไม่มี offset ในสตริง)
wind = pd.read_csv('../data/raw/wind_hourly_raw.csv', parse_dates=['datetime_utc'])
wind = wind[(wind.station_id == STATION) & (wind.wind_model == 'ecmwf_ifs')]
wind['datetime_utc'] = wind['datetime_utc'].dt.tz_localize('UTC')
wind = wind.set_index('datetime_utc')
data[WIND_VARS] = wind[WIND_VARS]

print(f'ช่วงเวลา: {data.index.min()} ถึง {data.index.max()}')
print(f'แถวเดิม {len(main):,} แถว -> เติมครบทุกชั่วโมงได้ {len(data):,} แถว')
display(data.notna().mean().mul(100).round(1).to_frame('% ชั่วโมงที่มีค่า'))

ช่วงเวลา: 2021-04-29 17:00:00+00:00 ถึง 2026-10-01 20:00:00+00:00
แถวเดิม 29,761 แถว -> เติมครบทุกชั่วโมงได้ 47,548 แถว


,% ชั่วโมงที่มีค่า
pm25,62.1
so2,44.9
pm25_bg,70.6
wind_speed_10m,100.0
wind_direction_10m,100.0
precipitation,100.0
temperature_2m,100.0
relative_humidity_2m,100.0


**อ่านผล:** เติมแถวจาก 29,761 เป็น 47,548 ชั่วโมง `asfreq('h')` ทำให้ 1 แถว = 1 ชั่วโมงพอดี จึงใช้ `shift` นับเป็นชั่วโมงจริงได้

PM2.5 มีค่า 62.1% และ SO2 44.9% ของชั่วโมงทั้งหมด ตรงกับความครบที่วัดไว้ใน Day 3 หัวข้อ 11 ส่วนลมมีครบ 100% เพราะดึงจากไฟล์ดิบ

## 5.2 สร้างฟีเจอร์

ฟีเจอร์ทุกตัวใช้ข้อมูล ณ เวลา t หรือก่อนหน้าเท่านั้น เพื่อกันข้อมูลจากอนาคตรั่วเข้ามา ซึ่งจะทำให้ผลดูดีเกินจริง

In [2]:
HORIZON = 24   # ทายล่วงหน้ากี่ชั่วโมง


def make_features(data, target):
    """สร้างฟีเจอร์ ณ เวลา t สำหรับทายค่า target ที่เวลา t+24 ใช้เฉพาะข้อมูล ณ เวลา t หรือก่อนหน้า"""
    f = pd.DataFrame(index=data.index)
    y = data[target]

    # ค่าย้อนหลังของตัวเอง lag0 คือค่า ณ เวลา t
    for lag in (0, 1, 2, 3, 24):
        f[f'lag{lag}'] = y.shift(lag)

    # ค่าเฉลี่ยย้อนหลัง นับรวมเวลา t ถอยไปในอดีตเท่านั้น
    for w in (6, 24):
        f[f'mean{w}'] = y.rolling(w, min_periods=w // 2).mean()

    # ลมและอากาศ ณ เวลา t (ไม่ใช้ของอนาคต เพราะตอนใช้งานจริงยังไม่รู้)
    rad = np.radians(data['wind_direction_10m'])
    f['wind_sin'] = np.sin(rad)
    f['wind_cos'] = np.cos(rad)
    f['wind_speed'] = data['wind_speed_10m']
    f['temp'] = data['temperature_2m']
    f['rh'] = data['relative_humidity_2m']
    f['rain24'] = data['precipitation'].rolling(24, min_periods=1).sum()

    # เวลาของชั่วโมงที่จะทาย รู้ล่วงหน้าได้เสมอ จึงไม่รั่ว
    t_target = (data.index + pd.Timedelta(hours=HORIZON)).tz_convert('Asia/Bangkok')
    f['hour'] = t_target.hour
    f['month'] = t_target.month
    f['dayofweek'] = t_target.dayofweek

    # สถานีพื้นหลัง ใช้เฉพาะ PM2.5 (SO2 ของ 717 หยุดปลายปี 2024)
    if target == 'pm25':
        f['bg_lag0'] = data['pm25_bg']
        f['bg_mean24'] = data['pm25_bg'].rolling(24, min_periods=12).mean()

    # คำตอบที่ต้องทาย คือค่าจริงที่เวลา t+24
    f['target'] = y.shift(-HORIZON)
    return f


feat = {'pm25': make_features(data, 'pm25'),
        'so2': make_features(data, 'so2')}

# เช็กว่าเลื่อนข้อมูลถูกทิศ: ดูชั่วโมงตัวอย่าง 1 ชั่วโมง (เวลา t)
t = pd.Timestamp('2021-05-05 22:00', tz='UTC')   # = 6 พ.ค. 2021 ตี 5 เวลาไทย
print('เวลา t                 :', t)
print('lag0  =', feat['pm25'].loc[t, 'lag0'], '| ค่าจริง ณ t      =', data.loc[t, 'pm25'])
print('lag24 =', feat['pm25'].loc[t, 'lag24'], '| ค่าจริง ณ t-24   =', data.loc[t - pd.Timedelta(hours=24), 'pm25'])
print('target=', feat['pm25'].loc[t, 'target'], '| ค่าจริง ณ t+24   =', data.loc[t + pd.Timedelta(hours=24), 'pm25'])
print()

rows = []
for p, f in feat.items():
    rows.append({'สาร': p,
                 'จำนวนฟีเจอร์': f.shape[1] - 1,
                 'แถวที่มีคำตอบ': f['target'].notna().sum(),
                 'แถวที่ไม่มีค่าว่างเลย': len(f.dropna())})
display(pd.DataFrame(rows).set_index('สาร'))

เวลา t                 : 2021-05-05 22:00:00+00:00
lag0  = 25.0 | ค่าจริง ณ t      = 25.0
lag24 = 13.0 | ค่าจริง ณ t-24   = 13.0
target= 14.0 | ค่าจริง ณ t+24   = 14.0



,จำนวนฟีเจอร์,แถวที่มีคำตอบ,แถวที่ไม่มีค่าว่างเลย
สาร,,,
pm25,18,29517,16016
so2,16,21338,9608


## 5.3 แบ่งข้อมูลตามเวลา

| ชุด | ช่วงเวลาของชั่วโมงที่ทาย (เวลาไทย) | ใช้ทำอะไร |
|---|---|---|
| train | ถึง 30 มิ.ย. 2024 | ให้โมเดลเรียน |
| valid | 1 ก.ค. ถึง 31 ธ.ค. 2024 | ปรับจูนโมเดล |
| test | 1 ม.ค. 2025 เป็นต้นไป | วัดผลจริง ใช้ครั้งเดียวตอนท้าย |

**ห้ามสุ่มแบ่ง:** ชั่วโมงที่อยู่ติดกันมีค่าใกล้กันมาก ถ้าสุ่ม ชั่วโมงในชุด test จะมีพี่น้องที่อยู่ติดกันอยู่ในชุด train ผลจะดีเกินจริง
ส่วนใช้งานจริงต้องทายอนาคตที่ไม่เคยเห็นเสมอ

**แบ่งตามเวลาของชั่วโมงที่ทาย (t+24) ไม่ใช่เวลา t:** ถ้าแบ่งตามเวลา t แถวท้าย ๆ ของ train จะมีคำตอบอยู่ในช่วง valid ไปแล้ว 24 ชม.
แบ่งตามเวลาที่ทายทำให้คำตอบของแต่ละชุดไม่ข้ามเส้นกันเลย ส่วนฟีเจอร์ของแถวต้น ๆ ใน valid ที่ใช้ค่าจากช่วง train ไม่ถือว่ารั่ว เพราะเป็นอดีตที่รู้อยู่แล้ว ณ เวลาที่ทาย

**ใช้เฉพาะแถวที่มีคำตอบ** แต่เก็บแถวที่ฟีเจอร์บางช่องว่างไว้ เพราะ LightGBM รับค่าว่างได้

In [3]:
TRAIN_END = pd.Timestamp('2024-07-01', tz='Asia/Bangkok')
VALID_END = pd.Timestamp('2025-01-01', tz='Asia/Bangkok')


def add_split(f):
    """เก็บเฉพาะแถวที่มีคำตอบ แล้วเพิ่มคอลัมน์ split บอกว่าแถวนี้อยู่ชุดไหน (แบ่งตามเวลาของชั่วโมงที่ทาย)"""
    f = f[f['target'].notna()].copy()
    t_target = f.index + pd.Timedelta(hours=HORIZON)
    f['split'] = 'test'
    f.loc[t_target < VALID_END, 'split'] = 'valid'
    f.loc[t_target < TRAIN_END, 'split'] = 'train'
    return f


feat['pm25'] = add_split(feat['pm25'])
feat['so2'] = add_split(feat['so2'])

counts = pd.DataFrame({'pm25': feat['pm25']['split'].value_counts(),
                       'so2': feat['so2']['split'].value_counts()})
display(counts.loc[['train', 'valid', 'test']])

,pm25,so2
split,,
train,18335,15474
valid,2696,2584
test,8486,3280


**อ่านผล:** รวมสามชุดได้ PM2.5 29,517 แถว (18,335 + 2,696 + 8,486) และ SO2 21,338 แถว (15,474 + 2,584 + 3,280) เท่ากับจำนวนแถวที่มีคำตอบในหัวข้อ 5.2 ไม่มีแถวหายหรือซ้ำ

แต่ละสารเก็บเป็นตารางเดียว มีคอลัมน์ `split` บอกชุด ขั้นต่อไปดึงชุดที่ต้องการด้วย `f[f.split == 'train']`

ข้อสังเกต
- ชุด valid เล็ก (ราว 2,600 แถว) เพราะเป็นช่วงครึ่งปี ใช้จูนได้แต่ตัวเลขจะแกว่งกว่าชุดอื่น
- ชุด test ของ SO2 มีแค่ 3,280 แถว น้อยกว่า PM2.5 (8,486 แถว) มาก ผลทดสอบของ SO2 จึงแกว่งได้มากกว่า

## 5.4 Baseline

baseline คือวิธีทายแบบง่ายที่สุด ถ้าโมเดลทายได้ไม่ดีกว่านี้ ก็ไม่คุ้มที่จะใช้โมเดล

| วิธี | ทายว่าพรุ่งนี้เวลาเดียวกันเท่ากับ |
|---|---|
| `persistence` | ค่าตอนนี้ (เวลา t) |
| `naive_week` | ค่าเวลาเดียวกันของสัปดาห์ก่อน (เวลา t+24−168 = t−144) |

`naive_week` ใช้สัปดาห์ก่อน ไม่ใช้เมื่อวาน เพราะทายล่วงหน้า 24 ชม. "เวลาเดียวกันของเมื่อวาน" ก็คือเวลา t ซึ่งซ้ำกับ `persistence`

**ตัววัดผล**
- MAE: ทายพลาดเฉลี่ยกี่หน่วย (µg/m³ สำหรับ PM2.5, ppm สำหรับ SO2)
- RMSE: คล้าย MAE แต่ลงโทษตอนพลาดหนัก ๆ มากกว่า

วัดบนชุด valid เท่านั้น (ชุด test เก็บไว้ใช้ครั้งเดียวตอนท้าย) และนับเฉพาะแถวที่ทั้งสองวิธีทายได้ เพื่อให้เทียบกันบนแถวชุดเดียวกัน

In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

rows = []
for p in ['pm25', 'so2']:
    valid = feat[p][feat[p]['split'] == 'valid'].copy()
    valid['persistence'] = valid['lag0']
    valid['naive_week'] = data[p].shift(144)          # ค่า ณ เวลา t-144
    valid = valid.dropna(subset=['target', 'persistence', 'naive_week'])

    for method in ['persistence', 'naive_week']:
        rows.append({'สาร': p,
                     'วิธี': method,
                     'จำนวนแถว': len(valid),
                     'MAE': mean_absolute_error(valid['target'], valid[method]),
                     'RMSE': root_mean_squared_error(valid['target'], valid[method])})

baseline_valid = pd.DataFrame(rows).set_index(['สาร', 'วิธี']).round(4)
display(baseline_valid)

จำนวนแถว     MAE    RMSE
สาร  วิธี                                 
pm25 persistence      1730  3.9881  5.5435
     naive_week       1730  5.7960  8.0454
so2  persistence      1514  0.0039  0.0074
     naive_week       1514  0.0047  0.0087

**อ่านผล:** `persistence` ชนะ `naive_week` ทั้งสองสาร ทั้ง MAE และ RMSE
- PM2.5: ทายพลาดเฉลี่ย 3.99 µg/m³ (`naive_week` พลาด 5.80)
- SO2: ทายพลาดเฉลี่ย 0.0039 ppm (`naive_week` พลาด 0.0047)

แปลว่าค่าของวันนี้บอกอะไรเกี่ยวกับพรุ่งนี้ได้มากกว่าค่าของสัปดาห์ก่อน **`persistence` จึงเป็นคู่แข่งหลักที่โมเดลต้องชนะ**

จำนวนแถวที่ใช้เทียบ (PM2.5 1,730 แถว, SO2 1,514 แถว) น้อยกว่าชุด valid ทั้งหมด (2,696 และ 2,584 แถว) เพราะนับเฉพาะแถวที่ทั้งสองวิธีทายได้
ตอนเทียบกับโมเดลต้องใช้แถวชุดเดียวกันนี้ด้วย

## 5.5 LightGBM

สอนโมเดลด้วยชุด train แล้ววัดผลบนชุด valid โดยใช้แถวชุดเดียวกับ baseline ในหัวข้อ 5.4

ใช้ early stopping คือให้โมเดลดูผลบนชุด valid ระหว่างสอน ถ้าทายไม่ดีขึ้นติดกัน 50 รอบให้หยุด ป้องกันไม่ให้โมเดลจำชุด train มากเกินไป
ข้อเสียคือผลบนชุด valid อาจดีกว่าความจริงเล็กน้อย เพราะ valid ถูกใช้ช่วยตัดสินใจแล้ว ผลที่เชื่อได้จริงคือชุด test ในหัวข้อถัดไป

In [5]:
import lightgbm as lgb

models = {}
rows = []
for p in ['pm25', 'so2']:
    f = feat[p]
    features = [c for c in f.columns if c not in ['target', 'split']]
    train = f[f['split'] == 'train']
    valid = f[f['split'] == 'valid']

    model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(train[features], train['target'],
              eval_X=valid[features], eval_y=valid['target'],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    models[p] = model
    print(f'{p}: สอนด้วย {len(train):,} แถว หยุดที่ต้นไม้ต้นที่ {model.best_iteration_}')

    # วัดผลบนแถวชุดเดียวกับ baseline (แถวที่ persistence และ naive_week ทายได้)
    v = valid.copy()
    v['naive_week'] = data[p].shift(144)
    v = v.dropna(subset=['target', 'lag0', 'naive_week'])
    pred = model.predict(v[features])
    rows.append({'สาร': p,
                 'วิธี': 'lightgbm',
                 'จำนวนแถว': len(v),
                 'MAE': mean_absolute_error(v['target'], pred),
                 'RMSE': root_mean_squared_error(v['target'], pred)})

model_valid = pd.DataFrame(rows).set_index(['สาร', 'วิธี']).round(4)
display(pd.concat([baseline_valid, model_valid]).sort_index())

pm25: สอนด้วย 18,335 แถว หยุดที่ต้นไม้ต้นที่ 98
so2: สอนด้วย 15,474 แถว หยุดที่ต้นไม้ต้นที่ 95


จำนวนแถว     MAE    RMSE
สาร  วิธี                                 
pm25 lightgbm         1730  3.5880  4.6190
     naive_week       1730  5.7960  8.0454
     persistence      1730  3.9881  5.5435
so2  lightgbm         1514  0.0037  0.0060
     naive_week       1514  0.0047  0.0087
     persistence      1514  0.0039  0.0074

**อ่านผล (ชุด valid):** LightGBM ทายพลาดน้อยกว่า `persistence` ทั้งสองสาร
- PM2.5: MAE 3.59 เทียบกับ 3.99 µg/m³ และ RMSE 4.62 เทียบกับ 5.54
- SO2: MAE 0.0037 เทียบกับ 0.0039 ppm และ RMSE 0.0060 เทียบกับ 0.0074

ทั้งสองสารหยุดสอนที่ต้นไม้ราว 100 ต้น (98 และ 95) จาก 1,000 ต้นที่ตั้งไว้ แปลว่าถ้าสอนต่อ ผลบนชุด valid จะไม่ดีขึ้นแล้ว

## 5.6 วัดผลจริงบนชุด test

ชุด test (ทายตั้งแต่ 1 ม.ค. 2025) ยังไม่เคยถูกใช้ตัดสินใจอะไรเลย จึงเป็นผลที่เชื่อได้ที่สุด เปิดใช้ครั้งเดียวตรงนี้
ใช้กติกาเดียวกับชุด valid คือนับเฉพาะแถวที่ทั้ง 3 วิธีทายได้

In [6]:
tests = {}
rows = []
for p in ['pm25', 'so2']:
    f = feat[p]
    features = [c for c in f.columns if c not in ['target', 'split']]
    test = f[f['split'] == 'test'].copy()
    test['persistence'] = test['lag0']
    test['naive_week'] = data[p].shift(144)
    test = test.dropna(subset=['target', 'persistence', 'naive_week'])
    test['lightgbm'] = models[p].predict(test[features])
    tests[p] = test

    for method in ['persistence', 'naive_week', 'lightgbm']:
        rows.append({'สาร': p,
                     'วิธี': method,
                     'จำนวนแถว': len(test),
                     'MAE': mean_absolute_error(test['target'], test[method]),
                     'RMSE': root_mean_squared_error(test['target'], test[method])})

result_test = pd.DataFrame(rows).set_index(['สาร', 'วิธี']).round(4)
display(result_test)

จำนวนแถว     MAE     RMSE
สาร  วิธี                                  
pm25 persistence      5756  4.6891   6.8883
     naive_week       5756  7.5542  11.3347
     lightgbm         5756  5.1338   7.6412
so2  persistence      1903  0.0065   0.0132
     naive_week       1903  0.0106   0.0206
     lightgbm         1903  0.0186   0.0315

**อ่านผล (ชุด test):** LightGBM แพ้ `persistence` ทั้งสองสาร ต่างจากชุด valid ที่ชนะ
- PM2.5: MAE 5.13 เทียบกับ 4.69 µg/m³
- SO2: MAE 0.0186 เทียบกับ 0.0065 ppm (แย่กว่าเกือบ 3 เท่า)

`naive_week` แพ้ `persistence` เหมือนชุด valid

แผนกำหนดไว้ว่าถ้าโมเดลไม่ชนะ baseline ให้เขียนตามจริงและหาสาเหตุ ดูหัวข้อ 5.7

## 5.7 ทำไมโมเดลแพ้บนชุด test

หัวข้อนี้ไม่ได้ปรับโมเดล แค่หาสาเหตุ เพราะถ้าปรับโมเดลจนชนะชุด test จะเท่ากับเห็นข้อสอบแล้วค่อยไปติว ผลจะไม่น่าเชื่อถือ

ดู 3 เรื่อง
1. ค่าในแต่ละชุดต่างกันแค่ไหน
2. SO2 ในและนอกช่วงที่ Day 3 ติดป้าย `unverifiable` (สงสัยว่าเซนเซอร์ดริฟต์) ซึ่ง Day 3 เขียนไว้แล้วว่าต้องรายงานทั้งแบบรวมและไม่รวมช่วงนี้
3. PM2.5 แยกตามฤดู

In [7]:
# 1) ค่าในแต่ละชุดต่างกันแค่ไหน
compare = pd.DataFrame({
    'PM2.5 ค่ากลาง': feat['pm25'].groupby('split')['target'].median(),
    'SO2 ค่ากลาง': feat['so2'].groupby('split')['target'].median(),
    'SO2 แถวที่เกิน 0.05 ppm': (feat['so2']['target'] > 0.05).groupby(feat['so2']['split']).sum(),
})
display(compare.loc[['train', 'valid', 'test']])

,PM2.5 ค่ากลาง,SO2 ค่ากลาง,SO2 แถวที่เกิน 0.05 ppm
split,,,
train,18.00,0.001,32
valid,14.80,0.001,2
test,21.95,0.003,666


**อ่านผล:** ชุด test ต่างจากชุดที่ใช้สอนชัดเจน
- PM2.5: ค่ากลางของ test 21.95 µg/m³ สูงกว่า train (18.00) และ valid (14.80)
- SO2: ชุด test มีแถวที่เกิน 0.05 ppm ถึง 666 แถว ขณะที่ train มีแค่ 32 แถว และ valid 2 แถว

โมเดลเรียนจากอดีต พอเจอค่าที่แทบไม่เคยเห็นตอนเรียนก็ทายได้ไม่ดี
ส่วน `persistence` ไม่ได้เรียนอะไร แค่ใช้ค่าล่าสุด จึงไม่โดนผลกระทบนี้

In [8]:
# 2) SO2 ในและนอกช่วงที่ Day 3 ติดป้าย unverifiable
episodes = pd.read_csv('../data/processed/episodes.csv', parse_dates=['start_utc', 'end_utc'])
drift = episodes[(episodes.station_id == STATION) & (episodes.pollutant == 'so2') & episodes.unverifiable]
display(drift[['start_utc', 'end_utc', 'duration_hours', 'peak_value']])

t = tests['so2']
t_target = t.index + pd.Timedelta(hours=HORIZON)
t['in_drift'] = False
for _, e in drift.iterrows():
    t.loc[(t_target >= e.start_utc) & (t_target <= e.end_utc), 'in_drift'] = True

rows = []
for in_drift, g in t.groupby('in_drift'):
    rows.append({'ช่วง': 'ในช่วง unverifiable' if in_drift else 'นอกช่วง unverifiable',
                 'จำนวนแถว': len(g),
                 'persistence MAE': mean_absolute_error(g['target'], g['persistence']),
                 'lightgbm MAE': mean_absolute_error(g['target'], g['lightgbm'])})
display(pd.DataFrame(rows).set_index('ช่วง').round(4))

,start_utc,end_utc,duration_hours,peak_value
1288,2025-03-15 11:00:00+00:00,2025-03-17 02:00:00+00:00,24,0.024
1347,2025-05-28 21:00:00+00:00,2025-06-19 03:00:00+00:00,385,0.109
1349,2025-06-19 14:00:00+00:00,2025-06-30 20:00:00+00:00,239,0.102


,จำนวนแถว,persistence MAE,lightgbm MAE
ช่วง,,,
นอกช่วง unverifiable,1513,0.0064,0.0068
ในช่วง unverifiable,390,0.0071,0.0644


**อ่านผล:** ที่ SO2 แพ้มากเกือบทั้งหมดมาจากช่วงที่ Day 3 ติดป้าย `unverifiable` ไว้ (3 เหตุการณ์ ส่วนใหญ่อยู่ช่วง 28 พ.ค. ถึง 30 มิ.ย. 2025 ค่าสูงสุด 0.109 และ 0.102 ppm)
- ในช่วงนั้น (390 แถว): LightGBM พลาดเฉลี่ย 0.0644 ppm ส่วน `persistence` พลาดแค่ 0.0071 เพราะค่าค้างอยู่ระดับเดิม การทายว่า "เท่าเดิม" จึงถูก
- นอกช่วงนั้น (1,513 แถว): สองวิธีใกล้กัน (0.0068 กับ 0.0064) แต่ `persistence` ยังดีกว่าเล็กน้อย

สรุปว่าถึงไม่นับช่วงที่สงสัยว่าเซนเซอร์ดริฟต์ LightGBM ก็ยังไม่ชนะ `persistence` สำหรับ SO2

In [9]:
# 3) PM2.5 แยกตามฤดู (นิยามฤดูเดียวกับ Day 3)
def season(ts):
    m, d = ts.month, ts.day
    if (m == 2 and d >= 15) or m in (3, 4) or (m == 5 and d < 15):
        return 'summer'
    if (m == 5 and d >= 15) or m in (6, 7, 8, 9) or (m == 10 and d < 15):
        return 'rainy'
    return 'winter'


t = tests['pm25']
t['season'] = (t.index + pd.Timedelta(hours=HORIZON)).tz_convert('Asia/Bangkok').map(season)

rows = []
for s, g in t.groupby('season'):
    rows.append({'ฤดู': s,
                 'จำนวนแถว': len(g),
                 'PM2.5 เฉลี่ย': g['target'].mean(),
                 'persistence MAE': mean_absolute_error(g['target'], g['persistence']),
                 'lightgbm MAE': mean_absolute_error(g['target'], g['lightgbm'])})
display(pd.DataFrame(rows).set_index('ฤดู').round(2))

# ชุด valid มีฤดูไหนบ้าง
valid = feat['pm25'][feat['pm25']['split'] == 'valid']
valid_season = (valid.index + pd.Timedelta(hours=HORIZON)).tz_convert('Asia/Bangkok').map(season)
print('ชุด valid ของ PM2.5 แยกตามฤดู:', valid_season.value_counts().to_dict())

,จำนวนแถว,PM2.5 เฉลี่ย,persistence MAE,lightgbm MAE
ฤดู,,,,
rainy,2913,18.65,3.57,3.59
summer,2473,28.18,5.45,6.34
winter,370,40.60,8.39,9.23


ชุด valid ของ PM2.5 แยกตามฤดู: {'rainy': 1647, 'winter': 1049}


**อ่านผล**
- ฤดูฝน: สองวิธีเกือบเท่ากัน (3.59 กับ 3.57)
- ฤดูร้อนและฤดูหนาวซึ่งฝุ่นสูง: LightGBM แย่กว่า (6.34 กับ 5.45 และ 9.23 กับ 8.39)
- ชุด valid มีแค่ฤดูฝน 1,647 แถว และฤดูหนาว 1,049 แถว ไม่มีฤดูร้อนเลย early stopping จึงเลือกจำนวนต้นไม้จากช่วงที่ฝุ่นไม่สูง

นี่เป็นจุดอ่อนของการแบ่งข้อมูลในหัวข้อ 5.3 คือชุด valid ครึ่งปีไม่ครอบคลุมทุกฤดู ถ้าทำใหม่ควรใช้ชุด valid ที่ครบทั้งปี
ไม่ได้แก้ในงานนี้ เพราะเป็นการเปลี่ยนหลังเห็นผลชุด test แล้ว

## สรุป Day 5

**สิ่งที่ทำ:** ทาย PM2.5 และ SO2 ล่วงหน้า 24 ชม. ที่สถานี 225611 ด้วย LightGBM เทียบกับ baseline 2 แบบ
แบ่งข้อมูลตามเวลา และฟีเจอร์ทุกตัวใช้ข้อมูล ณ เวลา t หรือก่อนหน้าเท่านั้น

**ผลหลัก (ชุด test ทาย ม.ค. 2025 ถึง ต.ค. 2026): LightGBM ไม่ชนะ `persistence`**

| สาร | persistence MAE | naive_week MAE | LightGBM MAE |
|---|---|---|---|
| PM2.5 (µg/m³) | 4.69 | 7.55 | 5.13 |
| SO2 (ppm) | 0.0065 | 0.0106 | 0.0186 |

**สาเหตุที่พบ**
1. ชุด test ต่างจากชุดที่ใช้สอน PM2.5 สูงกว่า และ SO2 มีค่าเกิน 0.05 ppm ถึง 666 แถว (train มี 32 แถว)
2. SO2 แพ้มากเพราะช่วงที่ Day 3 สงสัยว่าเซนเซอร์ดริฟต์ ไม่นับช่วงนั้นแล้วสองวิธีใกล้กัน แต่ `persistence` ยังดีกว่าเล็กน้อย
3. ชุด valid ไม่มีฤดูร้อน การเลือกจำนวนต้นไม้จึงไม่ได้ทดสอบกับช่วงฝุ่นสูง

**ข้อจำกัดที่ต้องเขียนใน README**
1. ชุด valid เป็นครึ่งปีหลังของ 2024 ไม่ครอบคลุมทุกฤดู
2. ใช้ลม ณ เวลา t เท่านั้น ไม่ได้ใช้ลมพยากรณ์ของวันพรุ่งนี้
3. ผลชุด test ของ SO2 มาจากปี 2025 เกือบทั้งหมด (ปี 2026 มีข้อมูลน้อยมาก) และรวมช่วงที่สงสัยว่าเซนเซอร์ดริฟต์
4. ไม่ได้จูน hyperparameter ใช้ค่าตั้งต้นของ LightGBM และ early stopping อย่างเดียว
5. ทำแค่สถานีเดียวและทายล่วงหน้า 24 ชม. อย่างเดียว

**สิ่งที่ควรลองถ้ามีเวลา** (ต้องตัดสินด้วยชุด valid ใหม่ ไม่ใช่ชุด test เดิม)
- ใช้ชุด valid ที่ครบทั้งปี
- ใช้ลมพยากรณ์ย้อนหลังจาก Open-Meteo Historical Forecast API ตามทางเลือกในแผน
- ทายล่วงหน้าสั้นลง (6 หรือ 12 ชม.)